# N=100, u_max=1280 top-peak refinement on bar GPU

This canonical boilerplate notebook refines the top five saved controls at the cap's established smoothness and sharpness. The monotone projected peak refiner uses per-member Armijo backtracking, repeats until three stable blocks, and has a four-hour elapsed limit.

In [1]:
from ofc.notebook_workflow import RunNotebook

run_name = "N100_u1280_top_peak_refinement_strict_gpu"
workflow = RunNotebook(run_name)

## Create the immutable config

Edit the arguments, then activate once. Returning later: leave `Activated=False`; the existing YAML is loaded without being changed. When `initialization_query` selects an old run, explicitly set `resume_optimizer: false` to reset Adam at the stored controls, or `resume_optimizer: true` to restore the matching stored Adam count and moments. Resuming requires exact, unperturbed `best` or `final` controls.

In [2]:
Activated = False

description = "N=100 u_max=1280 monotone backtracking refinement of the top five saved peak controls."
reuse_existing = False
parameters = {"N": 100, "t_interval": 4, "r_bg": -0.008716, "u_isbound": True, "v_isbound": True, "u_max": 1280, "v_max": 1000, "slew_limit": 0.05, "optimizer": "peak_refinement", "schedule": [[250, 1]], "adam_learning_rate": 0.1, "adam_beta1": 0.7, "adam_beta2": 0.99, "adam_eps": 1e-8, "lbfgs_history_size": 10, "lbfgs_max_linesearch_steps": 20, "lbfgs_tolerance": 0.000001, "peak_initial_step_size": 0.01, "peak_min_step_size": 1e-12, "peak_max_step_size": 0.1, "peak_backtracking_factor": 0.5, "peak_step_growth": 1.5, "peak_armijo": 0.0001, "peak_max_linesearch_steps": 24, "smoothness": 2.5e-7, "u_smooth": None, "v_smooth": None, "sharpness": 2.5e-8, "u_sharp": None, "v_sharp": None, "block_size": 25, "J_tol": 0.000001, "u_tol": 0.00001, "v_tol": 0.00001, "projected_gradient_tol": 0.00001, "projected_gradient_alpha": 1}
runtime = {"initialisations": 0, "fourier_num_modes": 5, "fourier_rms_amplitude": 0.3, "fourier_intensity_fraction": 0.3, "use_jit": True, "use_x64": True, "device": "gpu", "concurrent_workers": 1, "max_cases_per_batch": None, "max_initialisations_per_batch": None, "max_steps_per_chunk": 250, "max_batch_elapsed_seconds": 14400, "max_elapsed_seconds": 14400, "distribute_max_elapsed_across_batches": False, "repeat_schedule_until_stable": True, "auto_halt": True, "database": "results/results.sqlite3"}
initialization_query = {"where": {"status": ["complete", "failed"], "N": 100, "u_max": 1280, "smoothness": 2.5e-7, "sharpness": 2.5e-8}, "limit": 5, "order_by": "best_score", "descending": True, "control_kind": "best", "resume_optimizer": False, "perturbed": False}

config_document = workflow.create_config(
    activated=Activated,
    description=description,
    parameters=parameters,
    runtime=runtime,
    initialization_query=initialization_query,
    reuse_existing=reuse_existing,
)

Config creation is disabled (Activated=False).


## Run directly on `bar`'s GPU (detached)

This launches outside Slurm, verifies that JAX can see the GPU, and detaches the process into its own session and log so it survives a browser or laptop disconnect. The immutable config must use `device: auto` or `device: gpu`.

In [3]:
Activated = False

queue_id = None  # None creates a random ID; otherwise use a positive integer.
python_executable = None  # None uses this kernel's Python; otherwise give a path.
extra_arguments = []  # Example: ["--batch-index", "0"].
detached = True  # Keep running if the notebook/browser disconnects.
log_path = None  # None writes logs/<run-name>-local-<queue-id>.log.

active_queue_id = workflow.run_on_bar_gpu(
    activated=Activated,
    queue_id=queue_id,
    python_executable=python_executable,
    extra_arguments=extra_arguments,
    detached=detached,
    log_path=log_path,
)

Bar GPU execution is disabled (Activated=False).


## Submit through Slurm (alternative)

In [4]:
Activated = False

partition = "zen5,epyc"
time = "4-03:00:00"
cpus = 32
memory = "64G"
array = None  # None = array when multiple config batches; True or False overrides.
array_max_concurrent = None  # Example: 4; None leaves the array unthrottled.
job_name = None  # None uses run_name.
extra_arguments = []  # Extra sbatch arguments, e.g. ["--constraint=..."].

active_queue_id = workflow.submit_slurm(
    activated=Activated,
    partition=partition,
    time=time,
    cpus=cpus,
    memory=memory,
    array=array,
    array_max_concurrent=array_max_concurrent,
    job_name=job_name,
    extra_arguments=extra_arguments,
)

Slurm submission is disabled (Activated=False).


## Query persisted data

This is read-only and does not depend on executing any optional cell above. It supports any number of varying parameters; choose the dimensions used by each plot later. With config inheritance enabled, the existing YAML automatically supplies its database and immutable config identity (and the selected rows carry all resolved config parameters). Disable inheritance to query older runs using only database filters.

In [5]:
inherit_config = True
database = None
queue_id = None
config_run_rank = 1
statuses = None
filters = {}
sweep_parameters = ["initialization_index"]
require_saved_stage = True
limit = None
order_by = "run_id"
descending = False

query_result = workflow.query(
    inherit_config=inherit_config, database=database, queue_id=queue_id,
    config_run_rank=config_run_rank, statuses=statuses, filters=filters,
    sweep_parameters=sweep_parameters, require_saved_stage=require_saved_stage,
    limit=limit, order_by=order_by, descending=descending,
)

RuntimeError: Only 0 execution(s) match the inherited config 'N100_u1280_top_peak_refinement_strict_gpu'.

## Figure display and saving

In [ ]:
save_figure = None  # None = display only; "/" = figures/; or "/experiment/latest".
figure_format = "png"  # "png" or "pdf".
preview_dpi = 180  # Increase this if inline multi-sweep labels are hard to read.
save_dpi = 600  # Resolution used when saving PNG figures.

## Unified sweep summary

The history median/spread and objective strip use only the runs shown in each row. Dashed vertical lines mark learning-rate changes.


In [ ]:
sweep_parameter = None  # Required when the query selected multiple sweeps; e.g. "u_max".
history_points = 1200
summary_figure = query_result.plot_summary(
    sweep_parameter=sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    summary_figure,
    "01_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)


## Double sweep summary

In [ ]:
separate_sweep_parameter = "u_max"  # One rectangle per value.
colour_sweep_parameter = "adam_learning_rate"  # Coloured best trace/control per value.
history_points = 1200
double_sweep_figure = query_result.plot_double_sweep_summary(
    separate_sweep_parameter=separate_sweep_parameter,
    colour_sweep_parameter=colour_sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    double_sweep_figure,
    "02_double_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)

## Triple sweep summary

In [ ]:
row_sweep_parameter = "u_max"
column_sweep_parameter = "smoothness"
colour_sweep_parameter = "adam_learning_rate"
history_points = 1200
triple_sweep_figure = query_result.plot_triple_sweep_summary(
    row_sweep_parameter=row_sweep_parameter,
    column_sweep_parameter=column_sweep_parameter,
    colour_sweep_parameter=colour_sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    triple_sweep_figure,
    "03_triple_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)